In [1]:
# For four datasets
import os
import sys
import time
import glob
import random
import argparse

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from Helper_Scripts.calculate_metrics import (
    summarize_all_metrics,
    WINDOW as LOCAL_WINDOW,
    CWC_ETA,
)

from Helper_Scripts.models_TLMR import LSTMQuantileReg

# ============================================================
# GLOBAL CONFIG
# ============================================================
DATASET_PATHS = [
    "../Datasets/50Hertz.csv",
    "../Datasets/Amprion.csv",
    "../Datasets/TenneTTSO.csv",
    "../Datasets/TransnetBW.csv"

     
]




OUTPUT_CSV = "./results_all_methods_all_datasets.csv"
TRAJECTORY_DIR = "./trajectories/LSTM_0.80"
SAVE_TRAJECTORIES = True

SEQ_LEN    = 48
EPOCHS     = 200
LR         = 0.001
SEED       = 42
BATCH_SIZE = 32

ALPHA    = 0.1
TINIT    = 5000
SPLIT    = 0.5
GAMMA    = 0.001

DTACI_GAMMA_CANDIDATES = [0.001, 0.002, 0.004, 0.008, 0.0160]
DTACI_INTERVAL_SIZE    = 500

AGACI_GAMMA_CANDIDATES = [0.001, 0.002, 0.004, 0.008, 0.0160]
AGACI_EPS = 0.001

DOWNSAMPLE_STRIDE = 1

METHODS_TO_RUN = ["LACI","AgACI","DtACI"]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


# ============================================================
# MODEL: LSTM quantile regression
# ============================================================

def quantile_loss(y_true, y_pred, q):
    diff = y_true - y_pred
    return torch.mean(torch.maximum(q * diff, (q - 1) * diff))


# ============================================================
# DATA LOADING
# ============================================================
class SeqDataset(Dataset):
    def __init__(self, series, seq_len):
        self.X, self.y = [], []
        for i in range(len(series) - seq_len):
            self.X.append(series[i:i + seq_len])
            self.y.append(series[i + seq_len])
        self.X, self.y = np.array(self.X, dtype=np.float32), np.array(self.y, dtype=np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)


#For 4 datasets : 

def load_dataset(filepath, seq_len=96):
    df = pd.read_csv(filepath)
    y = df.iloc[:, 1:].values.flatten().astype(float)
    y = y[::4]

    scaler = StandardScaler()
    y_scaled = scaler.fit_transform(y.reshape(-1, 1)).flatten()

    full_dataset = SeqDataset(y_scaled, seq_len)

    return full_dataset, scaler, y, y_scaled



# ============================================================
# SHARED HELPERS
# ============================================================
def pad(arr, length):
    arr = np.array(arr, dtype=np.float32)
    if len(arr) < length:
        arr = np.concatenate([arr, np.full(length - len(arr), np.nan)])
    return arr


def scoring_function_v1(error_history, magnitude_history):
    if len(error_history) == 0:
        return 0.0
    n_hist = len(error_history)
    exponents = np.arange(n_hist - 1, -1, -1, dtype=np.float64)
    recency_w = 0.80 ** exponents
    mag_w = np.array(magnitude_history, dtype=np.float64) + 1e-3
    joint_w = recency_w * mag_w
    joint_w = joint_w / joint_w.sum()
    return float(np.dot(joint_w, error_history))


def pinball_loss(beta, theta, alpha=ALPHA):
    diff = beta - theta
    return alpha * diff - min(0.0, diff)


def beta_from_scores(score_t, scores_cal):
    beta_t = 1.0 - (np.sum(scores_cal <= score_t) / (len(scores_cal) + 1))
    return float(np.clip(beta_t, 0.0, 1.0))


def _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles):
    if t == tinit:
        current_epochs = EPOCHS
    elif (t - tinit) % 6 == 0:
        current_epochs = 1
    else:
        current_epochs = 0

    if current_epochs > 0:
        X_train_tensor = torch.tensor(X_train.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32, device=device)
        train_ds = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_ds, batch_size=min(BATCH_SIZE, len(train_ds)), shuffle=True)

        model_lower.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_lower(X_b), quantiles[0])
                opt_lower.zero_grad(); loss.backward(); opt_lower.step()

        model_upper.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_upper(X_b), quantiles[1])
                opt_upper.zero_grad(); loss.backward(); opt_upper.step()

    return current_epochs


def _fresh_models():
    model_lower = LSTMQuantileReg().to(device)
    model_upper = LSTMQuantileReg().to(device)
    opt_lower = torch.optim.Adam(model_lower.parameters(), lr=LR)
    opt_upper = torch.optim.Adam(model_upper.parameters(), lr=LR)
    return model_lower, model_upper, opt_lower, opt_upper





# ============================================================
# METHOD 2: ACI (Eq. 2/4, Gibbs & Candes 2021)
# ============================================================
def run_aci(full_dataset, alpha=ALPHA, gamma=GAMMA, tinit=TINIT, splitSize=SPLIT):
    T = len(full_dataset)
    n_pred = T - tinit
    alphaTrajectory = np.zeros(n_pred, dtype=np.float32)
    adaptErrSeq = np.zeros(n_pred, dtype=np.float32)
    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    alphat = alpha
    WINDOW_SIZE = tinit
    quantiles = [alpha / 2.0, 1.0 - alpha / 2.0]
    model_lower, model_upper, opt_lower, opt_upper = _fresh_models()

    for t in range(tinit, T):
        start_idx = t - WINDOW_SIZE
        current_window_idx = list(range(start_idx, t))
        n_train = int(round(splitSize * WINDOW_SIZE))
        trainPoints = current_window_idx[:n_train]
        calPoints = current_window_idx[n_train:]

        X_train = full_dataset.X[trainPoints].astype(np.float32)
        y_train = full_dataset.y[trainPoints].astype(np.float32)
        X_cal = full_dataset.X[calPoints].astype(np.float32)
        y_cal = full_dataset.y[calPoints].astype(np.float32)
        y_t = float(full_dataset.y[t])

        _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles)

        X_cal_tensor = torch.tensor(X_cal.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            model_lower.eval(); model_upper.eval()
            y_l_cal = model_lower(X_cal_tensor).cpu().numpy().reshape(-1)
            y_u_cal = model_upper(X_cal_tensor).cpu().numpy().reshape(-1)
        scores_cal = np.maximum(y_l_cal - y_cal, y_cal - y_u_cal)

        X_t_tensor = torch.tensor(full_dataset.X[t].reshape(1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            q_lo_t = float(model_lower(X_t_tensor).cpu().numpy()[0, 0])
            q_hi_t = float(model_upper(X_t_tensor).cpu().numpy()[0, 0])

        alphat = np.clip(alphat, 1e-6, 1 - 1e-6)
        level_adapt = np.clip((1.0 - alphat) * (1.0 + 1.0 / len(calPoints)), 0.0, 1.0)
        confQuantAdapt = float(np.quantile(scores_cal, level_adapt))

        y_l_adapt = q_lo_t - confQuantAdapt
        y_u_adapt = q_hi_t + confQuantAdapt
        y_range = np.max(y_train) - np.min(y_train)
        safety_margin = 5 * y_range
        y_l_adapt = np.clip(y_l_adapt, np.min(y_train) - safety_margin, np.max(y_train) + safety_margin)
        y_u_adapt = np.clip(y_u_adapt, np.min(y_train) - safety_margin, np.max(y_train) + safety_margin)

        err_t = float(not (y_l_adapt <= y_t <= y_u_adapt))
        alphat = np.clip(alphat + gamma * (alpha - err_t), 1e-6, 1 - 1e-6)

        idx = t - tinit
        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = err_t
        y_u_preds[idx] = y_u_adapt
        y_l_preds[idx] = y_l_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(f"[ACI] t={t}, alphat={alphat:.4f}, err_t={err_t:.0f}")

    return alphaTrajectory, adaptErrSeq, y_u_preds, y_l_preds, y_true_vals


# ============================================================
# METHOD 3: LACI (user's local variant)
# ============================================================
def run_laci(full_dataset, alpha=ALPHA, gamma=GAMMA, tinit=TINIT, splitSize=SPLIT):
    T = len(full_dataset)
    n_pred = T - tinit
    alphaTrajectory = np.zeros(n_pred, dtype=np.float32)
    adaptErrSeq = np.zeros(n_pred, dtype=np.float32)
    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    alphat = alpha
    WINDOW_SIZE = tinit
    quantiles = [alpha / 2.0, 1.0 - alpha / 2.0]
    error_history, magnitude_history = [], []
    model_lower, model_upper, opt_lower, opt_upper = _fresh_models()

    for t in range(tinit, T):
        start_idx = t - WINDOW_SIZE
        current_window_idx = list(range(start_idx, t))
        n_train = int(round(splitSize * WINDOW_SIZE))
        trainPoints = current_window_idx[:n_train]
        calPoints = current_window_idx[n_train:]

        X_train = full_dataset.X[trainPoints].astype(np.float32)
        y_train = full_dataset.y[trainPoints].astype(np.float32)
        X_cal = full_dataset.X[calPoints].astype(np.float32)
        y_cal = full_dataset.y[calPoints].astype(np.float32)
        y_t = float(full_dataset.y[t])

        _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles)

        X_cal_tensor = torch.tensor(X_cal.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            model_lower.eval(); model_upper.eval()
            y_l_cal = model_lower(X_cal_tensor).cpu().numpy().reshape(-1)
            y_u_cal = model_upper(X_cal_tensor).cpu().numpy().reshape(-1)
        scores_cal = np.maximum(y_l_cal - y_cal, y_cal - y_u_cal)

        alphat = np.clip(alphat, 1e-6, 1 - 1e-6)
        level_adapt = np.clip((1.0 - alphat) * (1.0 + 1.0 / len(calPoints)), 0.0, 1.0)
        confQuantAdapt = float(np.quantile(scores_cal, level_adapt))

        X_t_tensor = torch.tensor(full_dataset.X[t].reshape(1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            q_lo_t = float(model_lower(X_t_tensor).cpu().numpy()[0, 0])
            q_hi_t = float(model_upper(X_t_tensor).cpu().numpy()[0, 0])

        y_l_adapt = q_lo_t - confQuantAdapt
        y_u_adapt = q_hi_t + confQuantAdapt
        y_range = np.max(y_train) - np.min(y_train)
        safety_margin = 5 * y_range
        y_l_adapt = np.clip(y_l_adapt, np.min(y_train) - safety_margin, np.max(y_train) + safety_margin)
        y_u_adapt = np.clip(y_u_adapt, np.min(y_train) - safety_margin, np.max(y_train) + safety_margin)

        eval_idx = list(range(t - 23, t + 1))
        X_eval = full_dataset.X[eval_idx].astype(np.float32)
        y_eval = full_dataset.y[eval_idx].astype(np.float32)
        X_eval_tensor = torch.tensor(X_eval.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            y_l_eval = model_lower(X_eval_tensor).cpu().numpy().reshape(-1) - confQuantAdapt
            y_u_eval = model_upper(X_eval_tensor).cpu().numpy().reshape(-1) + confQuantAdapt
        lpicp = float(np.mean((y_eval >= y_l_eval) & (y_eval <= y_u_eval)))

        binary_err = float(not (y_l_adapt <= y_t <= y_u_adapt))
        magnitude = float(max((1.0 - alpha) - lpicp, 0.0))
        error_history.append(binary_err)
        magnitude_history.append(magnitude)
        weighted_err = scoring_function_v1(error_history, magnitude_history)

        alphat = np.clip(alphat + gamma * (alpha - weighted_err), 1e-6, 1 - 1e-6)

        idx = t - tinit
        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = binary_err
        y_u_preds[idx] = y_u_adapt
        y_l_preds[idx] = y_l_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(f"[LACI] t={t}, alphat={alphat:.4f}, weighted_err={weighted_err:.4f}")

    return alphaTrajectory, adaptErrSeq, y_u_preds, y_l_preds, y_true_vals


# ============================================================
# METHOD 4: DtACI
# ============================================================
def run_dtaci(full_dataset, alpha=ALPHA, tinit=TINIT, splitSize=SPLIT,
              gamma_candidates=DTACI_GAMMA_CANDIDATES, interval_size=DTACI_INTERVAL_SIZE):
    T = len(full_dataset)
    n_pred = T - tinit
    k = len(gamma_candidates)
    gammas = np.array(gamma_candidates, dtype=np.float64)

    sigma = 1.0 / (2.0 * interval_size)
    eta = (1.0 / interval_size) * np.sqrt(
        3.0 * (np.log(2 * k * interval_size) + 1.0) / ((1.0 - alpha) ** 2 * alpha ** 2)
    )

    alphaTrajectory = np.zeros(n_pred, dtype=np.float32)
    adaptErrSeq = np.zeros(n_pred, dtype=np.float32)
    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    WINDOW_SIZE = tinit
    quantiles = [alpha / 2.0, 1.0 - alpha / 2.0]
    expert_alphas = np.full(k, alpha, dtype=np.float64)
    expert_weights = np.ones(k, dtype=np.float64) / k
    model_lower, model_upper, opt_lower, opt_upper = _fresh_models()

    for t in range(tinit, T):
        start_idx = t - WINDOW_SIZE
        current_window_idx = list(range(start_idx, t))
        n_train = int(round(splitSize * WINDOW_SIZE))
        trainPoints = current_window_idx[:n_train]
        calPoints = current_window_idx[n_train:]

        X_train = full_dataset.X[trainPoints].astype(np.float32)
        y_train = full_dataset.y[trainPoints].astype(np.float32)
        X_cal = full_dataset.X[calPoints].astype(np.float32)
        y_cal = full_dataset.y[calPoints].astype(np.float32)
        y_t = float(full_dataset.y[t])

        _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles)

        X_cal_tensor = torch.tensor(X_cal.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            model_lower.eval(); model_upper.eval()
            y_l_cal = model_lower(X_cal_tensor).cpu().numpy().reshape(-1)
            y_u_cal = model_upper(X_cal_tensor).cpu().numpy().reshape(-1)
        scores_cal = np.maximum(y_l_cal - y_cal, y_cal - y_u_cal)

        X_t_tensor = torch.tensor(full_dataset.X[t].reshape(1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            q_lo_t = float(model_lower(X_t_tensor).cpu().numpy()[0, 0])
            q_hi_t = float(model_upper(X_t_tensor).cpu().numpy()[0, 0])

        y_range = np.max(y_train) - np.min(y_train)
        safety_margin = 5 * y_range
        score_t = max(q_lo_t - y_t, y_t - q_hi_t)

        expert_alphas_clipped = np.clip(expert_alphas, 1e-6, 1 - 1e-6)
        expert_err = np.zeros(k, dtype=np.float64)
        for i in range(k):
            level_i = np.clip((1.0 - expert_alphas_clipped[i]) * (1.0 + 1.0 / len(calPoints)), 0.0, 1.0)
            confQuant_i = float(np.quantile(scores_cal, level_i))
            y_l_i = np.clip(q_lo_t - confQuant_i, np.min(y_train) - safety_margin, np.max(y_train) + safety_margin)
            y_u_i = np.clip(q_hi_t + confQuant_i, np.min(y_train) - safety_margin, np.max(y_train) + safety_margin)
            expert_err[i] = float(not (y_l_i <= y_t <= y_u_i))
        beta_t = beta_from_scores(score_t, scores_cal)

        p_t = expert_weights / expert_weights.sum()
        alphat = float(np.dot(p_t, expert_alphas_clipped))

        level_adapt = np.clip((1.0 - alphat) * (1.0 + 1.0 / len(calPoints)), 0.0, 1.0)
        confQuantAdapt = float(np.quantile(scores_cal, level_adapt))
        y_l_adapt = np.clip(q_lo_t - confQuantAdapt, np.min(y_train) - safety_margin, np.max(y_train) + safety_margin)
        y_u_adapt = np.clip(q_hi_t + confQuantAdapt, np.min(y_train) - safety_margin, np.max(y_train) + safety_margin)
        err_t = float(not (y_l_adapt <= y_t <= y_u_adapt))

        losses = np.array([pinball_loss(beta_t, a, alpha) for a in expert_alphas_clipped])
        w_bar = expert_weights * np.exp(-eta * losses)
        W_bar = w_bar.sum()
        expert_weights = (1.0 - sigma) * w_bar + W_bar * sigma / k

        expert_alphas = np.clip(expert_alphas + gammas * (alpha - expert_err), -1.0, 2.0)

        idx = t - tinit
        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = err_t
        y_u_preds[idx] = y_u_adapt
        y_l_preds[idx] = y_l_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(f"[DtACI] t={t}, alphat={alphat:.4f}, err_t={err_t:.0f}")

    return alphaTrajectory, adaptErrSeq, y_u_preds, y_l_preds, y_true_vals


# ============================================================
# METHOD 5: AgACI
# ============================================================
def run_agaci(full_dataset, alpha=ALPHA, tinit=TINIT, splitSize=SPLIT,
              gamma_candidates=AGACI_GAMMA_CANDIDATES, eps=AGACI_EPS):
    T = len(full_dataset)
    n_pred = T - tinit
    k = len(gamma_candidates)
    gammas = np.array(gamma_candidates, dtype=np.float64)

    alphaTrajectory = np.zeros(n_pred, dtype=np.float32)
    adaptErrSeq = np.zeros(n_pred, dtype=np.float32)
    y_u_preds = np.zeros(n_pred, dtype=np.float32)
    y_l_preds = np.zeros(n_pred, dtype=np.float32)
    y_true_vals = np.zeros(n_pred, dtype=np.float32)

    WINDOW_SIZE = tinit
    quantiles = [alpha / 2.0, 1.0 - alpha / 2.0]

    expertAlphas = np.full(k, alpha, dtype=np.float64)
    expertProbs = np.full(k, 1.0 / k, dtype=np.float64)
    expertSqLosses = np.zeros(k, dtype=np.float64)
    expertEtas = np.zeros(k, dtype=np.float64)
    expertLValues = np.zeros(k, dtype=np.float64)
    expertMaxLosses = np.zeros(k, dtype=np.float64)

    model_lower, model_upper, opt_lower, opt_upper = _fresh_models()

    for t in range(tinit, T):
        start_idx = t - WINDOW_SIZE
        current_window_idx = list(range(start_idx, t))
        n_train = int(round(splitSize * WINDOW_SIZE))
        trainPoints = current_window_idx[:n_train]
        calPoints = current_window_idx[n_train:]

        X_train = full_dataset.X[trainPoints].astype(np.float32)
        y_train = full_dataset.y[trainPoints].astype(np.float32)
        X_cal = full_dataset.X[calPoints].astype(np.float32)
        y_cal = full_dataset.y[calPoints].astype(np.float32)
        y_t = float(full_dataset.y[t])

        _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles)

        X_cal_tensor = torch.tensor(X_cal.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            model_lower.eval(); model_upper.eval()
            y_l_cal = model_lower(X_cal_tensor).cpu().numpy().reshape(-1)
            y_u_cal = model_upper(X_cal_tensor).cpu().numpy().reshape(-1)
        scores_cal = np.maximum(y_l_cal - y_cal, y_cal - y_u_cal)

        X_t_tensor = torch.tensor(full_dataset.X[t].reshape(1, SEQ_LEN), dtype=torch.float32, device=device)
        with torch.no_grad():
            q_lo_t = float(model_lower(X_t_tensor).cpu().numpy()[0, 0])
            q_hi_t = float(model_upper(X_t_tensor).cpu().numpy()[0, 0])

        y_range = np.max(y_train) - np.min(y_train)
        safety_margin = 5 * y_range
        score_t = max(q_lo_t - y_t, y_t - q_hi_t)
        beta_t = beta_from_scores(score_t, scores_cal)

        alphat = float(np.dot(expertProbs, expertAlphas))
        err_t = float(alphat > beta_t)

        expertLosses = (err_t - alpha) * (expertAlphas - alphat)
        expertSqLosses = expertSqLosses + expertLosses ** 2
        expertMaxLosses = np.maximum(expertMaxLosses, np.abs(expertLosses))
        expertEVals = 2.0 ** (np.ceil(np.log2(np.abs(expertMaxLosses) + eps)) + 1.0)
        expertLValues = expertLValues + 0.5 * (
            expertLosses * (1.0 + expertEtas * expertLosses)
            + expertEVals * (expertEtas * expertLosses > 0.5).astype(np.float64)
        )
        with np.errstate(divide="ignore", invalid="ignore"):
            candidate_eta = np.sqrt(np.log(k) / expertSqLosses)
        candidate_eta = np.where(expertSqLosses > 0, candidate_eta, np.inf)
        expertEtas = np.minimum(1.0 / expertEVals, candidate_eta)

        expertAlphas = expertAlphas + gammas * (alpha - (expertAlphas > beta_t).astype(np.float64))

        eta_L = expertEtas * expertLValues
        expertWeights = expertEtas * np.exp(-eta_L + np.max(eta_L))
        expertProbs = expertWeights / expertWeights.sum()

        alphat_clip = np.clip(alphat, 1e-6, 1 - 1e-6)
        level_adapt = np.clip((1.0 - alphat_clip) * (1.0 + 1.0 / len(calPoints)), 0.0, 1.0)
        confQuantAdapt = float(np.quantile(scores_cal, level_adapt))
        y_l_adapt = np.clip(q_lo_t - confQuantAdapt, np.min(y_train) - safety_margin, np.max(y_train) + safety_margin)
        y_u_adapt = np.clip(q_hi_t + confQuantAdapt, np.min(y_train) - safety_margin, np.max(y_train) + safety_margin)

        idx = t - tinit
        alphaTrajectory[idx] = alphat
        adaptErrSeq[idx] = err_t
        y_u_preds[idx] = y_u_adapt
        y_l_preds[idx] = y_l_adapt
        y_true_vals[idx] = y_t

        if t % 200 == 0:
            print(f"[AgACI] t={t}, alphat={alphat:.4f}, err_t={err_t:.0f}")

    return alphaTrajectory, adaptErrSeq, y_u_preds, y_l_preds, y_true_vals


METHOD_FUNCS = {
   
    "ACI": run_aci,
    "LACI": run_laci,
    "DtACI": run_dtaci,
    "AgACI": run_agaci,
}


# ============================================================
# DRIVER
# ============================================================
def run_one_method(dataset_name, method_name, full_dataset, scaler, **kwargs):
    print(f"\n{'=' * 70}\n{dataset_name} :: {method_name}\n{'=' * 70}")
    run_fn = METHOD_FUNCS[method_name]
    start_time = time.time()
    alphaTraj, errSeq, y_u_preds, y_l_preds, y_true_vals = run_fn(full_dataset, **kwargs)
    time_taken = time.time() - start_time

    y_u_inv = scaler.inverse_transform(y_u_preds.reshape(-1, 1)).flatten()
    y_l_inv = scaler.inverse_transform(y_l_preds.reshape(-1, 1)).flatten()
    y_true_inv = scaler.inverse_transform(y_true_vals.reshape(-1, 1)).flatten()

    metrics = summarize_all_metrics(
        y_true_inv, y_l_inv, y_u_inv,
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA,
        time_taken=time_taken,
    )

    print(f"[{dataset_name}/{method_name}] " +
          " | ".join(f"{k}={v:.4f}" for k, v in metrics.items()
                      if isinstance(v, float) and k != "time_sec"))
    print(f"[{dataset_name}/{method_name}] time_sec={time_taken:.2f}")

    if SAVE_TRAJECTORIES:
        os.makedirs(TRAJECTORY_DIR, exist_ok=True)
        max_len = max(len(alphaTraj), len(errSeq), len(y_u_inv), len(y_l_inv), len(y_true_inv))
        traj_df = pd.DataFrame({
            "step": np.arange(max_len),
            "alphaTrajectory": pad(alphaTraj, max_len),
            "errSeq": pad(errSeq, max_len),
            "y_u_pred": pad(y_u_inv, max_len),
            "y_l_pred": pad(y_l_inv, max_len),
            "y_true": pad(y_true_inv, max_len),
        })
        safe_ds = os.path.splitext(os.path.basename(dataset_name))[0]
        traj_df.to_csv(os.path.join(TRAJECTORY_DIR, f"{safe_ds}__{method_name}.csv"), index=False)

    return metrics


def main(dataset_paths):
    if not dataset_paths:
        print("No datasets provided. Edit DATASET_PATHS at the top of this "
              "script, or pass paths with --data path1.csv path2.csv ...")
        sys.exit(1)

    all_rows = []
    for dataset_path in dataset_paths:
        print(f"\n\n########## Loading dataset: {dataset_path} ##########")
        full_dataset, scaler, y_raw, y_scaled = load_dataset(dataset_path, seq_len=SEQ_LEN)
        print(f"Series length: {len(y_raw)}, usable (t - seq_len) pairs: {len(full_dataset)}")

        if len(full_dataset) <= TINIT:
            print(f"WARNING: dataset {dataset_path} has fewer than TINIT={TINIT} "
                  f"usable points ({len(full_dataset)}). Skipping.")
            continue

        for method_name in METHODS_TO_RUN:
            kwargs = dict(alpha=ALPHA, tinit=TINIT, splitSize=SPLIT)
            if method_name == "Normal_QR":
                kwargs = dict(alpha=ALPHA, tinit=TINIT, splitSize=1)
            elif method_name in ("ACI", "LACI"):
                kwargs["gamma"] = GAMMA
            elif method_name == "DtACI":
                kwargs["gamma_candidates"] = DTACI_GAMMA_CANDIDATES
                kwargs["interval_size"] = DTACI_INTERVAL_SIZE
            elif method_name == "AgACI":
                kwargs["gamma_candidates"] = AGACI_GAMMA_CANDIDATES
                kwargs["eps"] = AGACI_EPS

            metrics = run_one_method(dataset_path, method_name, full_dataset, scaler, **kwargs)
            row = {"dataset": os.path.basename(dataset_path), "method": method_name}
            row.update(metrics)
            all_rows.append(row)

            pd.DataFrame(all_rows).to_csv(OUTPUT_CSV, index=False)

    results_df = pd.DataFrame(all_rows)
    col_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                 "LMPIW", "LCWC", "STD_LE", "n_local_windows", "time_sec"]
    col_order = [c for c in col_order if c in results_df.columns]
    results_df = results_df[col_order + [c for c in results_df.columns if c not in col_order]]
    results_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\nAll results written to: {OUTPUT_CSV}")
    if SAVE_TRAJECTORIES:
        print(f"Per-step trajectories written to: {TRAJECTORY_DIR}/")
    print("\n", results_df.to_string(index=False))
    return results_df


if __name__ == "__main__":
    dataset_paths = DATASET_PATHS
    main(dataset_paths)

device: cuda


########## Loading dataset: ../Datasets/50Hertz.csv ##########
Series length: 9528, usable (t - seq_len) pairs: 9480

../Datasets/50Hertz.csv :: LACI
[LACI] t=5000, alphat=0.1001, weighted_err=0.0000
[LACI] t=5200, alphat=0.0945, weighted_err=0.0120
[LACI] t=5400, alphat=0.0944, weighted_err=0.6472
[LACI] t=5600, alphat=0.0952, weighted_err=0.1845
[LACI] t=5800, alphat=0.0937, weighted_err=0.2631
[LACI] t=6000, alphat=0.0988, weighted_err=0.4000
[LACI] t=6200, alphat=0.1084, weighted_err=0.0000
[LACI] t=6400, alphat=0.1166, weighted_err=0.0001
[LACI] t=6600, alphat=0.1159, weighted_err=0.1307
[LACI] t=6800, alphat=0.1102, weighted_err=0.0089
[LACI] t=7000, alphat=0.1065, weighted_err=0.0000
[LACI] t=7200, alphat=0.1165, weighted_err=0.0056
[LACI] t=7400, alphat=0.1214, weighted_err=0.8682
[LACI] t=7600, alphat=0.0974, weighted_err=0.2308
[LACI] t=7800, alphat=0.0893, weighted_err=0.5050
[LACI] t=8000, alphat=0.0985, weighted_err=0.0006
[LACI] t=8200, alphat=0.0873, weigh

In [2]:
# For two locations



DATASET_PATHS = [
    "../Datasets/Location1.csv",
    "../Datasets/Location2.csv"
]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

TINIT    = 5000


# ============================================================
# DATA LOADING
# ============================================================
class SeqDataset(Dataset):
    def __init__(self, series, seq_len):
        self.X, self.y = [], []
        for i in range(len(series) - seq_len):
            self.X.append(series[i:i + seq_len])
            self.y.append(series[i + seq_len])
        self.X, self.y = np.array(self.X, dtype=np.float32), np.array(self.y, dtype=np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)




# For 2 locations : 


def load_dataset(filepath, seq_len=96):
    # Load data
    df = pd.read_csv(filepath)
    df = df[:10000]

   
    


    y = df["Power"].to_numpy(dtype=float)

    
    # Scale
    scaler = StandardScaler()
    y_scaled = scaler.fit_transform(y.reshape(-1, 1)).flatten()

    # Create dataset
    full_dataset = SeqDataset(y_scaled, seq_len)

    return full_dataset, scaler, y, y_scaled



# ============================================================
# SHARED HELPERS
# ============================================================
def pad(arr, length):
    arr = np.array(arr, dtype=np.float32)
    if len(arr) < length:
        arr = np.concatenate([arr, np.full(length - len(arr), np.nan)])
    return arr




def pinball_loss(beta, theta, alpha=ALPHA):
    diff = beta - theta
    return alpha * diff - min(0.0, diff)


def beta_from_scores(score_t, scores_cal):
    beta_t = 1.0 - (np.sum(scores_cal <= score_t) / (len(scores_cal) + 1))
    return float(np.clip(beta_t, 0.0, 1.0))


def _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles):
    if t == tinit:
        current_epochs = EPOCHS
    elif (t - tinit) % 6 == 0:
        current_epochs = 1
    else:
        current_epochs = 0

    if current_epochs > 0:
        X_train_tensor = torch.tensor(X_train.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32, device=device)
        train_ds = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_ds, batch_size=min(BATCH_SIZE, len(train_ds)), shuffle=True)

        model_lower.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_lower(X_b), quantiles[0])
                opt_lower.zero_grad(); loss.backward(); opt_lower.step()

        model_upper.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_upper(X_b), quantiles[1])
                opt_upper.zero_grad(); loss.backward(); opt_upper.step()

    return current_epochs


def _fresh_models():
    model_lower = LSTMQuantileReg().to(device)
    model_upper = LSTMQuantileReg().to(device)
    opt_lower = torch.optim.Adam(model_lower.parameters(), lr=LR)
    opt_upper = torch.optim.Adam(model_upper.parameters(), lr=LR)
    return model_lower, model_upper, opt_lower, opt_upper





METHOD_FUNCS = {
  
    "ACI": run_aci,
    "LACI": run_laci,
    "DtACI": run_dtaci,
    "AgACI": run_agaci,
}


# ============================================================
# DRIVER
# ============================================================
def run_one_method(dataset_name, method_name, full_dataset, scaler, **kwargs):
    print(f"\n{'=' * 70}\n{dataset_name} :: {method_name}\n{'=' * 70}")
    run_fn = METHOD_FUNCS[method_name]
    start_time = time.time()
    alphaTraj, errSeq, y_u_preds, y_l_preds, y_true_vals = run_fn(full_dataset, **kwargs)
    time_taken = time.time() - start_time

    y_u_inv = scaler.inverse_transform(y_u_preds.reshape(-1, 1)).flatten()
    y_l_inv = scaler.inverse_transform(y_l_preds.reshape(-1, 1)).flatten()
    y_true_inv = scaler.inverse_transform(y_true_vals.reshape(-1, 1)).flatten()

    metrics = summarize_all_metrics(
        y_true_inv, y_l_inv, y_u_inv,
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA,
        time_taken=time_taken,
    )

    print(f"[{dataset_name}/{method_name}] " +
          " | ".join(f"{k}={v:.4f}" for k, v in metrics.items()
                      if isinstance(v, float) and k != "time_sec"))
    print(f"[{dataset_name}/{method_name}] time_sec={time_taken:.2f}")

    if SAVE_TRAJECTORIES:
        os.makedirs(TRAJECTORY_DIR, exist_ok=True)
        max_len = max(len(alphaTraj), len(errSeq), len(y_u_inv), len(y_l_inv), len(y_true_inv))
        traj_df = pd.DataFrame({
            "step": np.arange(max_len),
            "alphaTrajectory": pad(alphaTraj, max_len),
            "errSeq": pad(errSeq, max_len),
            "y_u_pred": pad(y_u_inv, max_len),
            "y_l_pred": pad(y_l_inv, max_len),
            "y_true": pad(y_true_inv, max_len),
        })
        safe_ds = os.path.splitext(os.path.basename(dataset_name))[0]
        traj_df.to_csv(os.path.join(TRAJECTORY_DIR, f"{safe_ds}__{method_name}.csv"), index=False)

    return metrics


def main(dataset_paths):
    if not dataset_paths:
        print("No datasets provided. Edit DATASET_PATHS at the top of this "
              "script, or pass paths with --data path1.csv path2.csv ...")
        sys.exit(1)

    all_rows = []
    for dataset_path in dataset_paths:
        print(f"\n\n########## Loading dataset: {dataset_path} ##########")
        full_dataset, scaler, y_raw, y_scaled = load_dataset(dataset_path, seq_len=SEQ_LEN)
        print(f"Series length: {len(y_raw)}, usable (t - seq_len) pairs: {len(full_dataset)}")

        if len(full_dataset) <= TINIT:
            print(f"WARNING: dataset {dataset_path} has fewer than TINIT={TINIT} "
                  f"usable points ({len(full_dataset)}). Skipping.")
            continue

        for method_name in METHODS_TO_RUN:
            kwargs = dict(alpha=ALPHA, tinit=TINIT, splitSize=SPLIT)
            if method_name == "Normal_QR":
                kwargs = dict(alpha=ALPHA, tinit=TINIT, splitSize=1)
            elif method_name in ("ACI", "LACI"):
                kwargs["gamma"] = GAMMA
            elif method_name == "DtACI":
                kwargs["gamma_candidates"] = DTACI_GAMMA_CANDIDATES
                kwargs["interval_size"] = DTACI_INTERVAL_SIZE
            elif method_name == "AgACI":
                kwargs["gamma_candidates"] = AGACI_GAMMA_CANDIDATES
                kwargs["eps"] = AGACI_EPS

            metrics = run_one_method(dataset_path, method_name, full_dataset, scaler, **kwargs)
            row = {"dataset": os.path.basename(dataset_path), "method": method_name}
            row.update(metrics)
            all_rows.append(row)

            pd.DataFrame(all_rows).to_csv(OUTPUT_CSV, index=False)

    results_df = pd.DataFrame(all_rows)
    col_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                 "LMPIW", "LCWC", "STD_LE", "n_local_windows", "time_sec"]
    col_order = [c for c in col_order if c in results_df.columns]
    results_df = results_df[col_order + [c for c in results_df.columns if c not in col_order]]
    results_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\nAll results written to: {OUTPUT_CSV}")
    if SAVE_TRAJECTORIES:
        print(f"Per-step trajectories written to: {TRAJECTORY_DIR}/")
    print("\n", results_df.to_string(index=False))
    return results_df


if __name__ == "__main__":
    dataset_paths = DATASET_PATHS
    main(dataset_paths)

device: cuda


########## Loading dataset: ../Datasets/Location1.csv ##########
Series length: 10000, usable (t - seq_len) pairs: 9952

../Datasets/Location1.csv :: LACI
[LACI] t=5000, alphat=0.1001, weighted_err=0.0000
[LACI] t=5200, alphat=0.1075, weighted_err=0.0000
[LACI] t=5400, alphat=0.0905, weighted_err=0.6388
[LACI] t=5600, alphat=0.0830, weighted_err=0.3244
[LACI] t=5800, alphat=0.0897, weighted_err=0.0168
[LACI] t=6000, alphat=0.0805, weighted_err=0.9104
[LACI] t=6200, alphat=0.0723, weighted_err=0.0372
[LACI] t=6400, alphat=0.0656, weighted_err=0.0006
[LACI] t=6600, alphat=0.0711, weighted_err=0.0417
[LACI] t=6800, alphat=0.0650, weighted_err=0.0088
[LACI] t=7000, alphat=0.0383, weighted_err=0.0010
[LACI] t=7200, alphat=0.0502, weighted_err=0.0002
[LACI] t=7400, alphat=0.0503, weighted_err=0.1024
[LACI] t=7600, alphat=0.0462, weighted_err=0.1794
[LACI] t=7800, alphat=0.0398, weighted_err=0.0383
[LACI] t=8000, alphat=0.0423, weighted_err=0.0000
[LACI] t=8200, alphat=0.0476, 

In [3]:

DATASET_PATHS = [
    "../Datasets/Plant_1_Generation_Data.csv",
    "../Datasets/Plant_2_Generation_Data.csv"

     
]



TINIT    = 4000






device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)





# ============================================================
# DATA LOADING
# ============================================================
class SeqDataset(Dataset):
    def __init__(self, series, seq_len):
        self.X, self.y = [], []
        for i in range(len(series) - seq_len):
            self.X.append(series[i:i + seq_len])
            self.y.append(series[i + seq_len])
        self.X, self.y = np.array(self.X, dtype=np.float32), np.array(self.y, dtype=np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)




# For Solar datasets : 

def load_dataset(filepath, seq_len=96):
    # Load data
    df = pd.read_csv(filepath)
    df = df[:10000]

    # Extract meter readings and drop NaN values
    print(df.columns)
    y = df.loc[df["DAILY_YIELD"].notna() & (df["DAILY_YIELD"] != 0), "DAILY_YIELD"].to_numpy(dtype=float)

    # Optional: Downsample every 4th reading
    # y = y[::4]

    # Scale
    scaler = StandardScaler()
    y_scaled = scaler.fit_transform(y.reshape(-1, 1)).flatten()

    # Create dataset
    full_dataset = SeqDataset(y_scaled, seq_len)

    return full_dataset, scaler, y, y_scaled

# ============================================================
# SHARED HELPERS
# ============================================================
def pad(arr, length):
    arr = np.array(arr, dtype=np.float32)
    if len(arr) < length:
        arr = np.concatenate([arr, np.full(length - len(arr), np.nan)])
    return arr





def pinball_loss(beta, theta, alpha=ALPHA):
    diff = beta - theta
    return alpha * diff - min(0.0, diff)


def beta_from_scores(score_t, scores_cal):
    beta_t = 1.0 - (np.sum(scores_cal <= score_t) / (len(scores_cal) + 1))
    return float(np.clip(beta_t, 0.0, 1.0))


def _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles):
    if t == tinit:
        current_epochs = EPOCHS
    elif (t - tinit) % 6 == 0:
        current_epochs = 1
    else:
        current_epochs = 0

    if current_epochs > 0:
        X_train_tensor = torch.tensor(X_train.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32, device=device)
        train_ds = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_ds, batch_size=min(BATCH_SIZE, len(train_ds)), shuffle=True)

        model_lower.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_lower(X_b), quantiles[0])
                opt_lower.zero_grad(); loss.backward(); opt_lower.step()

        model_upper.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_upper(X_b), quantiles[1])
                opt_upper.zero_grad(); loss.backward(); opt_upper.step()

    return current_epochs


def _fresh_models():
    model_lower = LSTMQuantileReg().to(device)
    model_upper = LSTMQuantileReg().to(device)
    opt_lower = torch.optim.Adam(model_lower.parameters(), lr=LR)
    opt_upper = torch.optim.Adam(model_upper.parameters(), lr=LR)
    return model_lower, model_upper, opt_lower, opt_upper




METHOD_FUNCS = {
   
    "ACI": run_aci,
    "LACI": run_laci,
    "DtACI": run_dtaci,
    "AgACI": run_agaci,
}


# ============================================================
# DRIVER
# ============================================================
def run_one_method(dataset_name, method_name, full_dataset, scaler, **kwargs):
    print(f"\n{'=' * 70}\n{dataset_name} :: {method_name}\n{'=' * 70}")
    run_fn = METHOD_FUNCS[method_name]
    start_time = time.time()
    alphaTraj, errSeq, y_u_preds, y_l_preds, y_true_vals = run_fn(full_dataset, **kwargs)
    time_taken = time.time() - start_time

    y_u_inv = scaler.inverse_transform(y_u_preds.reshape(-1, 1)).flatten()
    y_l_inv = scaler.inverse_transform(y_l_preds.reshape(-1, 1)).flatten()
    y_true_inv = scaler.inverse_transform(y_true_vals.reshape(-1, 1)).flatten()

    metrics = summarize_all_metrics(
        y_true_inv, y_l_inv, y_u_inv,
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA,
        time_taken=time_taken,
    )

    print(f"[{dataset_name}/{method_name}] " +
          " | ".join(f"{k}={v:.4f}" for k, v in metrics.items()
                      if isinstance(v, float) and k != "time_sec"))
    print(f"[{dataset_name}/{method_name}] time_sec={time_taken:.2f}")

    if SAVE_TRAJECTORIES:
        os.makedirs(TRAJECTORY_DIR, exist_ok=True)
        max_len = max(len(alphaTraj), len(errSeq), len(y_u_inv), len(y_l_inv), len(y_true_inv))
        traj_df = pd.DataFrame({
            "step": np.arange(max_len),
            "alphaTrajectory": pad(alphaTraj, max_len),
            "errSeq": pad(errSeq, max_len),
            "y_u_pred": pad(y_u_inv, max_len),
            "y_l_pred": pad(y_l_inv, max_len),
            "y_true": pad(y_true_inv, max_len),
        })
        safe_ds = os.path.splitext(os.path.basename(dataset_name))[0]
        traj_df.to_csv(os.path.join(TRAJECTORY_DIR, f"{safe_ds}__{method_name}.csv"), index=False)

    return metrics


def main(dataset_paths):
    if not dataset_paths:
        print("No datasets provided. Edit DATASET_PATHS at the top of this "
              "script, or pass paths with --data path1.csv path2.csv ...")
        sys.exit(1)

    all_rows = []
    for dataset_path in dataset_paths:
        print(f"\n\n########## Loading dataset: {dataset_path} ##########")
        full_dataset, scaler, y_raw, y_scaled = load_dataset(dataset_path, seq_len=SEQ_LEN)
        print(f"Series length: {len(y_raw)}, usable (t - seq_len) pairs: {len(full_dataset)}")

        if len(full_dataset) <= TINIT:
            print(f"WARNING: dataset {dataset_path} has fewer than TINIT={TINIT} "
                  f"usable points ({len(full_dataset)}). Skipping.")
            continue

        for method_name in METHODS_TO_RUN:
            kwargs = dict(alpha=ALPHA, tinit=TINIT, splitSize=SPLIT)
            if method_name == "Normal_QR":
                kwargs = dict(alpha=ALPHA, tinit=TINIT, splitSize=1)
            elif method_name in ("ACI", "LACI"):
                kwargs["gamma"] = GAMMA
            elif method_name == "DtACI":
                kwargs["gamma_candidates"] = DTACI_GAMMA_CANDIDATES
                kwargs["interval_size"] = DTACI_INTERVAL_SIZE
            elif method_name == "AgACI":
                kwargs["gamma_candidates"] = AGACI_GAMMA_CANDIDATES
                kwargs["eps"] = AGACI_EPS

            metrics = run_one_method(dataset_path, method_name, full_dataset, scaler, **kwargs)
            row = {"dataset": os.path.basename(dataset_path), "method": method_name}
            row.update(metrics)
            all_rows.append(row)

            pd.DataFrame(all_rows).to_csv(OUTPUT_CSV, index=False)

    results_df = pd.DataFrame(all_rows)
    col_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                 "LMPIW", "LCWC", "STD_LE", "n_local_windows", "time_sec"]
    col_order = [c for c in col_order if c in results_df.columns]
    results_df = results_df[col_order + [c for c in results_df.columns if c not in col_order]]
    results_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\nAll results written to: {OUTPUT_CSV}")
    if SAVE_TRAJECTORIES:
        print(f"Per-step trajectories written to: {TRAJECTORY_DIR}/")
    print("\n", results_df.to_string(index=False))
    return results_df


if __name__ == "__main__":
    dataset_paths = DATASET_PATHS
    main(dataset_paths)

device: cuda


########## Loading dataset: ../Datasets/Plant_1_Generation_Data.csv ##########
Index(['DATE_TIME', 'PLANT_ID', 'SOURCE_KEY', 'DC_POWER', 'AC_POWER',
       'DAILY_YIELD', 'TOTAL_YIELD'],
      dtype='object')
Series length: 7379, usable (t - seq_len) pairs: 7331

../Datasets/Plant_1_Generation_Data.csv :: LACI
[LACI] t=4000, alphat=0.0991, weighted_err=1.0000
[LACI] t=4200, alphat=0.0207, weighted_err=0.3807
[LACI] t=4400, alphat=0.0196, weighted_err=0.0215
[LACI] t=4600, alphat=0.0259, weighted_err=0.0000
[LACI] t=4800, alphat=0.0459, weighted_err=0.0000
[LACI] t=5000, alphat=0.0659, weighted_err=0.0000
[LACI] t=5200, alphat=0.0859, weighted_err=0.0000
[LACI] t=5400, alphat=0.1059, weighted_err=0.0000
[LACI] t=5600, alphat=0.1259, weighted_err=0.0000
[LACI] t=5800, alphat=0.1351, weighted_err=0.0169
[LACI] t=6000, alphat=0.0785, weighted_err=0.4425
[LACI] t=6200, alphat=0.0728, weighted_err=0.0000
[LACI] t=6400, alphat=0.0928, weighted_err=0.0000
[LACI] t=6600, alphat=0

In [4]:


# ============================================================
# GLOBAL CONFIG
# ============================================================
DATASET_PATHS = [
    "../Datasets/hydropower_dataset.csv",
]

TINIT    = 200





device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)




# ============================================================
# DATA LOADING
# ============================================================
class SeqDataset(Dataset):
    def __init__(self, series, seq_len):
        self.X, self.y = [], []
        for i in range(len(series) - seq_len):
            self.X.append(series[i:i + seq_len])
            self.y.append(series[i + seq_len])
        self.X, self.y = np.array(self.X, dtype=np.float32), np.array(self.y, dtype=np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype=torch.float32)





# For HydroPower : 

def load_dataset(filepath, seq_len=96):
    # Load data
    df = pd.read_csv(filepath)

   
    df["total_power_output"] = df["Actual Day Peak (MW)"] + df["Actual Eve Peak (MW)"]
    y = df.loc[(df["total_power_output"].notna()) & (df["total_power_output"] != 0), "total_power_output"].to_numpy(dtype=float)
    

    # Scale
    scaler = StandardScaler()
    y_scaled = scaler.fit_transform(y.reshape(-1, 1)).flatten()

    # Create dataset
    full_dataset = SeqDataset(y_scaled, seq_len)

    return full_dataset, scaler, y, y_scaled



# ============================================================
# SHARED HELPERS
# ============================================================
def pad(arr, length):
    arr = np.array(arr, dtype=np.float32)
    if len(arr) < length:
        arr = np.concatenate([arr, np.full(length - len(arr), np.nan)])
    return arr





def pinball_loss(beta, theta, alpha=ALPHA):
    diff = beta - theta
    return alpha * diff - min(0.0, diff)


def beta_from_scores(score_t, scores_cal):
    beta_t = 1.0 - (np.sum(scores_cal <= score_t) / (len(scores_cal) + 1))
    return float(np.clip(beta_t, 0.0, 1.0))


def _maybe_train(t, tinit, X_train, y_train, model_lower, model_upper, opt_lower, opt_upper, quantiles):
    if t == tinit:
        current_epochs = EPOCHS
    elif (t - tinit) % 6 == 0:
        current_epochs = 1
    else:
        current_epochs = 0

    if current_epochs > 0:
        X_train_tensor = torch.tensor(X_train.reshape(-1, SEQ_LEN), dtype=torch.float32, device=device)
        y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32, device=device)
        train_ds = TensorDataset(X_train_tensor, y_train_tensor)
        train_loader = DataLoader(train_ds, batch_size=min(BATCH_SIZE, len(train_ds)), shuffle=True)

        model_lower.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_lower(X_b), quantiles[0])
                opt_lower.zero_grad(); loss.backward(); opt_lower.step()

        model_upper.train()
        for _ in range(current_epochs):
            for X_b, y_b in train_loader:
                loss = quantile_loss(y_b, model_upper(X_b), quantiles[1])
                opt_upper.zero_grad(); loss.backward(); opt_upper.step()

    return current_epochs


def _fresh_models():
    model_lower = LSTMQuantileReg().to(device)
    model_upper = LSTMQuantileReg().to(device)
    opt_lower = torch.optim.Adam(model_lower.parameters(), lr=LR)
    opt_upper = torch.optim.Adam(model_upper.parameters(), lr=LR)
    return model_lower, model_upper, opt_lower, opt_upper





METHOD_FUNCS = {
   
    "ACI": run_aci,
    "LACI": run_laci,
    "DtACI": run_dtaci,
    "AgACI": run_agaci,
}


# ============================================================
# DRIVER
# ============================================================
def run_one_method(dataset_name, method_name, full_dataset, scaler, **kwargs):
    print(f"\n{'=' * 70}\n{dataset_name} :: {method_name}\n{'=' * 70}")
    run_fn = METHOD_FUNCS[method_name]
    start_time = time.time()
    alphaTraj, errSeq, y_u_preds, y_l_preds, y_true_vals = run_fn(full_dataset, **kwargs)
    time_taken = time.time() - start_time

    y_u_inv = scaler.inverse_transform(y_u_preds.reshape(-1, 1)).flatten()
    y_l_inv = scaler.inverse_transform(y_l_preds.reshape(-1, 1)).flatten()
    y_true_inv = scaler.inverse_transform(y_true_vals.reshape(-1, 1)).flatten()

    metrics = summarize_all_metrics(
        y_true_inv, y_l_inv, y_u_inv,
        alpha=ALPHA, w=LOCAL_WINDOW, eta=CWC_ETA,
        time_taken=time_taken,
    )

    print(f"[{dataset_name}/{method_name}] " +
          " | ".join(f"{k}={v:.4f}" for k, v in metrics.items()
                      if isinstance(v, float) and k != "time_sec"))
    print(f"[{dataset_name}/{method_name}] time_sec={time_taken:.2f}")

    if SAVE_TRAJECTORIES:
        os.makedirs(TRAJECTORY_DIR, exist_ok=True)
        max_len = max(len(alphaTraj), len(errSeq), len(y_u_inv), len(y_l_inv), len(y_true_inv))
        traj_df = pd.DataFrame({
            "step": np.arange(max_len),
            "alphaTrajectory": pad(alphaTraj, max_len),
            "errSeq": pad(errSeq, max_len),
            "y_u_pred": pad(y_u_inv, max_len),
            "y_l_pred": pad(y_l_inv, max_len),
            "y_true": pad(y_true_inv, max_len),
        })
        safe_ds = os.path.splitext(os.path.basename(dataset_name))[0]
        traj_df.to_csv(os.path.join(TRAJECTORY_DIR, f"{safe_ds}__{method_name}.csv"), index=False)

    return metrics


def main(dataset_paths):
    if not dataset_paths:
        print("No datasets provided. Edit DATASET_PATHS at the top of this "
              "script, or pass paths with --data path1.csv path2.csv ...")
        sys.exit(1)

    all_rows = []
    for dataset_path in dataset_paths:
        print(f"\n\n########## Loading dataset: {dataset_path} ##########")
        full_dataset, scaler, y_raw, y_scaled = load_dataset(dataset_path, seq_len=SEQ_LEN)
        print(f"Series length: {len(y_raw)}, usable (t - seq_len) pairs: {len(full_dataset)}")

        if len(full_dataset) <= TINIT:
            print(f"WARNING: dataset {dataset_path} has fewer than TINIT={TINIT} "
                  f"usable points ({len(full_dataset)}). Skipping.")
            continue

        for method_name in METHODS_TO_RUN:
            kwargs = dict(alpha=ALPHA, tinit=TINIT, splitSize=SPLIT)
            if method_name == "Normal_QR":
                kwargs = dict(alpha=ALPHA, tinit=TINIT, splitSize=1)
            elif method_name in ("ACI", "LACI"):
                kwargs["gamma"] = GAMMA
            elif method_name == "DtACI":
                kwargs["gamma_candidates"] = DTACI_GAMMA_CANDIDATES
                kwargs["interval_size"] = DTACI_INTERVAL_SIZE
            elif method_name == "AgACI":
                kwargs["gamma_candidates"] = AGACI_GAMMA_CANDIDATES
                kwargs["eps"] = AGACI_EPS

            metrics = run_one_method(dataset_path, method_name, full_dataset, scaler, **kwargs)
            row = {"dataset": os.path.basename(dataset_path), "method": method_name}
            row.update(metrics)
            all_rows.append(row)

            pd.DataFrame(all_rows).to_csv(OUTPUT_CSV, index=False)

    results_df = pd.DataFrame(all_rows)
    col_order = ["dataset", "method", "PICP", "MPIW", "MLPICP", "LPICE", "LPIA",
                 "LMPIW", "LCWC", "STD_LE", "n_local_windows", "time_sec"]
    col_order = [c for c in col_order if c in results_df.columns]
    results_df = results_df[col_order + [c for c in results_df.columns if c not in col_order]]
    results_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\nAll results written to: {OUTPUT_CSV}")
    if SAVE_TRAJECTORIES:
        print(f"Per-step trajectories written to: {TRAJECTORY_DIR}/")
    print("\n", results_df.to_string(index=False))
    return results_df


if __name__ == "__main__":
    dataset_paths = DATASET_PATHS
    main(dataset_paths)

device: cuda


########## Loading dataset: ../Datasets/hydropower_dataset.csv ##########
Series length: 1753, usable (t - seq_len) pairs: 1705

../Datasets/hydropower_dataset.csv :: LACI
[LACI] t=200, alphat=0.1001, weighted_err=0.0000
[LACI] t=400, alphat=0.0868, weighted_err=0.2002
[LACI] t=600, alphat=0.0632, weighted_err=0.3999
[LACI] t=800, alphat=0.0735, weighted_err=0.0269
[LACI] t=1000, alphat=0.0460, weighted_err=0.0011
[LACI] t=1200, alphat=0.0388, weighted_err=0.1180
[LACI] t=1400, alphat=0.0374, weighted_err=0.0155
[LACI] t=1600, alphat=0.0511, weighted_err=0.0015
[../Datasets/hydropower_dataset.csv/LACI] PICP=0.9003 | MPIW=161.7550 | MLPICP=0.9028 | LPICE=0.0506 | LPIA=0.6856 | LMPIW=161.5815 | LCWC=581697544888602624.0000 | STD_LE=0.1297
[../Datasets/hydropower_dataset.csv/LACI] time_sec=9.05

../Datasets/hydropower_dataset.csv :: AgACI
[AgACI] t=200, alphat=0.1000, err_t=0
[AgACI] t=400, alphat=0.0913, err_t=0
[AgACI] t=600, alphat=0.0386, err_t=0
[AgACI] t=800, alphat=0